<a href="https://colab.research.google.com/github/EUNTELLA/baseball/blob/main/test0810.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import shutil
from pathlib import Path

repo_path = "/content/baseball"
# Corrected URL for downloading the main branch as a zip file
repo_zip_url = "https://github.com/EUNTELLA/baseball/archive/refs/heads/main.zip"

# Remove the directory if it already exists to ensure a clean setup
if Path(repo_path).exists():
    shutil.rmtree(repo_path)
    print(f"Removed existing directory: {repo_path}")

# Create the directory
Path(repo_path).mkdir(parents=True, exist_ok=True)

# Download the repository as a zip file
zip_filename = Path(repo_path) / "baseball_main.zip"
!wget -O "{zip_filename}" "{repo_zip_url}"

# Unzip the contents
# Ensure the zip file exists before attempting to unzip
if zip_filename.exists():
    !unzip -o "{zip_filename}" -d "{repo_path}"

    # The content will be extracted into a subdirectory named 'baseball-main'
    # Move contents up to the 'baseball' directory
    extracted_path = Path(repo_path) / "baseball-main"
    if extracted_path.exists():
        for item in extracted_path.iterdir():
            shutil.move(str(item), repo_path)
        shutil.rmtree(extracted_path)
    else:
        print(f"Warning: Extracted directory '{extracted_path}' not found.")
else:
    print(f"Error: Zip file not downloaded from {repo_zip_url}.")

# Change into the repository directory
# Only change directory if the repo_path exists and seems to contain the repo
if Path(repo_path).exists() and any(Path(repo_path).iterdir()):
    %cd "{repo_path}"
else:
    print(f"Error: Repository directory '{repo_path}' is empty or does not exist.")

--2026-08-10 02:13:04--  https://github.com/EUNTELLA/baseball/archive/refs/heads/main.zip
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/EUNTELLA/baseball/zip/refs/heads/main [following]
--2026-08-10 02:13:04--  https://codeload.github.com/EUNTELLA/baseball/zip/refs/heads/main
Resolving codeload.github.com (codeload.github.com)... 20.205.243.165
Connecting to codeload.github.com (codeload.github.com)|20.205.243.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘/content/baseball/baseball_main.zip’

/content/baseball/b     [ <=>                ]   3.78M  20.3MB/s    in 0.2s    

2026-08-10 02:13:04 (20.3 MB/s) - ‘/content/baseball/baseball_main.zip’ saved [3964001]

Archive:  /content/baseball/baseball_main.zip
353b96e010a4d7a8d8a7742953dd8090b83d2a2b
   cr

In [4]:
!mkdir -p /content/baseball/open
!unzip -o "/content/drive/MyDrive/open.zip" -d /content/baseball/open

Archive:  /content/drive/MyDrive/open.zip
  inflating: /content/baseball/open/baseline_submit.zip  
  inflating: /content/baseball/open/data/sample_submission.csv  
  inflating: /content/baseball/open/data/test.csv  
  inflating: /content/baseball/open/data/trackman_history.csv  
  inflating: /content/baseball/open/data/train.csv  
  inflating: /content/baseball/open/data_description.md  


In [5]:
import sys
from pathlib import Path

# The common.py module is now located inside /content/baseball/0826
experiment_dir = Path("/content/baseball/0826")

# Add the experiment directory to sys.path
if str(experiment_dir) not in sys.path:
    sys.path.insert(0, str(experiment_dir))

print((experiment_dir / "common.py").exists())
print(Path("/content/baseball/open/data/train.csv").exists())

True
True


In [6]:
# This cell's functionality for sys.path is now handled by cell EO3-WBQSUT0W.
# No action needed here.

In [7]:
"""운영진 RandomForest를 2022~2024 expanding-window Fold로 평가한다."""

import joblib
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

from common import (
    BASE_CAT_COLS,
    RESULTS_DIR,
    TARGET_COL,
    VALID_YEARS,
    Timer,
    brier_metrics,
    load_train,
    print_metrics,
    save_json,
)


def build_model(features: list[str]) -> Pipeline:
    numeric_cols = [c for c in features if c not in BASE_CAT_COLS]
    preprocessor = ColumnTransformer(
        [
            (
                "cat",
                OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
                BASE_CAT_COLS,
            ),
            ("num", SimpleImputer(strategy="median"), numeric_cols),
        ]
    )
    return Pipeline(
        [
            ("pre", preprocessor),
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=100,
                    max_depth=10,
                    min_samples_leaf=200,
                    n_jobs=-1,
                    random_state=42,
                ),
            ),
        ]
    )


def main() -> None:
    train, features = load_train()
    predictions, targets, years = [], [], []
    fold_results = []

    for valid_year in VALID_YEARS:
        train_mask = train["season"] < valid_year
        valid_mask = train["season"] == valid_year
        model = build_model(features)
        with Timer() as timer:
            model.fit(train.loc[train_mask, features], train.loc[train_mask, TARGET_COL])
            pred = model.predict_proba(train.loc[valid_mask, features])[:, 1]
        y = train.loc[valid_mask, TARGET_COL].to_numpy()
        metrics = brier_metrics(y, pred)
        metrics.update({"valid_year": valid_year, "seconds": timer.seconds})
        fold_results.append(metrics)
        print_metrics(str(valid_year), metrics)
        predictions.append(pred.astype(np.float32))
        targets.append(y.astype(np.int8))
        years.append(np.full(len(y), valid_year, dtype=np.int16))

    all_pred = np.concatenate(predictions)
    all_y = np.concatenate(targets)
    all_year = np.concatenate(years)
    overall = brier_metrics(all_y, all_pred)
    print_metrics("OOF 전체", overall)

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        RESULTS_DIR / "01_rf_oof.npz", y=all_y, prediction=all_pred, year=all_year
    )
    save_json(
        RESULTS_DIR / "01_rf_metrics.json",
        {"model": "RandomForest baseline", "folds": fold_results, "overall": overall},
    )
    joblib.dump(model, RESULTS_DIR / "01_rf_last_fold.pkl", compress=3)


if __name__ == "__main__":
    main()



2022: n=247,472, Brier=0.24398726, Skill=0.020775, Score=2077.49, actual=0.52892, pred=0.53032
2023: n=245,525, Brier=0.25165133, Skill=-0.006605, Score=0.00, actual=0.49996, pred=0.52026
2024: n=253,507, Brier=0.24876879, Skill=0.004156, Score=415.57, actual=0.48610, pred=0.50120
OOF 전체: n=746,504, Brier=0.24813174, Skill=0.007379, Score=737.95, actual=0.50485, pred=0.51712


In [8]:
import json

with open(
    "/content/baseball/0826/results/01_rf_metrics.json",
    encoding="utf-8",
) as file:
    result = json.load(file)

result

{'model': 'RandomForest baseline',
 'folds': [{'n': 247472,
   'target_rate': 0.5289204435249241,
   'prediction_mean': 0.5303156179131836,
   'brier': 0.24398725988122968,
   'brier_skill': 0.020774896092398687,
   'competition_score': 2077.4896092398685,
   'valid_year': 2022,
   'seconds': 167.22409878800005},
  {'n': 245525,
   'target_rate': 0.49995723449750534,
   'prediction_mean': 0.5202615226861995,
   'brier': 0.2516513260021733,
   'brier_skill': -0.0066053113725677015,
   'competition_score': 0.0,
   'valid_year': 2023,
   'seconds': 225.95782405599994},
  {'n': 253507,
   'target_rate': 0.4861049201797189,
   'prediction_mean': 0.5011994317753441,
   'brier': 0.2487687945941139,
   'brier_skill': 0.004155738098026385,
   'competition_score': 415.5738098026385,
   'valid_year': 2024,
   'seconds': 291.9603206400001}],
 'overall': {'n': 746504,
  'target_rate': 0.5048546290441847,
  'prediction_mean': 0.5171211959954393,
  'brier': 0.24813174182170644,
  'brier_skill': 0.007

In [9]:
from pathlib import Path

print(Path("0826/02_catboost_time_cv.ipynb").exists())
print(Path("0826/common.py").exists())
print(Path("open/data/train.csv").exists())

True
True
True


In [10]:
%pip install -q catboost==1.2.10
%run 0826/02_catboost_time_cv.ipynb

0:	learn: 0.2491444	test: 0.2494628	best: 0.2494628 (0)	total: 3.13s	remaining: 26m
100:	learn: 0.2419468	test: 0.2452984	best: 0.2449608 (42)	total: 3m 18s	remaining: 13m 3s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 0.2449608364
bestIteration = 42

Shrink model to first 43 iterations.
2022: n=247,472, Brier=0.24496084, Skill=0.016868, Score=1686.75, actual=0.52892, pred=0.52434
0:	learn: 0.2490975	test: 0.2499689	best: 0.2499689 (0)	total: 2.65s	remaining: 22m 3s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 0.2499689271
bestIteration = 0

Shrink model to first 1 iterations.
2023: n=245,525, Brier=0.24996893, Skill=0.000124, Score=12.43, actual=0.49996, pred=0.50033
0:	learn: 0.2492731	test: 0.2498492	best: 0.2498492 (0)	total: 3.44s	remaining: 28m 39s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 0.2497143768
bestIteration = 1

Shrink model to first 2 iterations.
2024: n=253,507, Brier=0.24971438, Skill=0.000370, Score=3

In [11]:
import json

with open(
    "0826/results/02_catboost_metrics.json",
    encoding="utf-8",
) as file:
    catboost_result = json.load(file)

catboost_result

{'model': 'CatBoost categorical',
 'params': {'iterations': 500,
  'depth': 8,
  'learning_rate': 0.08,
  'loss_function': 'Logloss',
  'eval_metric': 'BrierScore',
  'l2_leaf_reg': 5.0,
  'random_seed': 42,
  'thread_count': -1,
  'verbose': 100,
  'allow_writing_files': False},
 'folds': [{'n': 247472,
   'target_rate': 0.5289204435249241,
   'prediction_mean': 0.524342382246087,
   'brier': 0.24496083641803185,
   'brier_skill': 0.016867517543714627,
   'competition_score': 1686.7517543714628,
   'valid_year': 2022,
   'seconds': 249.3001893899998,
   'best_iteration': 42},
  {'n': 245525,
   'target_rate': 0.49995723449750534,
   'prediction_mean': 0.5003262561772126,
   'brier': 0.2499689270683197,
   'brier_skill': 0.00012428441207745777,
   'competition_score': 12.428441207745777,
   'valid_year': 2023,
   'seconds': 214.51711361000002,
   'best_iteration': 0},
  {'n': 253507,
   'target_rate': 0.4861049201797189,
   'prediction_mean': 0.49930002095334763,
   'brier': 0.24971437

In [13]:
%run 0826/03_probability_calibration.ipynb

2023 원본: n=245,525, Brier=0.24996893, Skill=0.000124, Score=12.43, actual=0.49996, pred=0.50033
2023 보정: n=245,525, Brier=0.25004508, Skill=-0.000180, Score=0.00, actual=0.49996, pred=0.49221
2024 원본: n=253,507, Brier=0.24971438, Skill=0.000370, Score=37.05, actual=0.48610, pred=0.49930
2024 보정: n=253,507, Brier=0.24955876, Skill=0.000993, Score=99.35, actual=0.48610, pred=0.49521
최종 재학습용 alpha=1.331431, rate=0.504855


In [12]:
!mkdir -p "/content/drive/MyDrive/baseball-results"
!cp -r 0826/results/. "/content/drive/MyDrive/baseball-results/"

In [14]:
%cd /content/baseball
%run 0826/04_feature_engineering_catboost.ipynb

/content/baseball
0:	learn: 0.2493483	test: 0.2495944	best: 0.2495944 (0)	total: 2.76s	remaining: 32m 8s
100:	learn: 0.2419301	test: 0.2448368	best: 0.2446693 (61)	total: 4m 39s	remaining: 27m 39s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.2446693222
bestIteration = 61

Shrink model to first 62 iterations.
2022: n=247,472, Brier=0.24466932, Skill=0.018037, Score=1803.75, actual=0.52892, pred=0.52394
0:	learn: 0.2493101	test: 0.2499600	best: 0.2499600 (0)	total: 3.75s	remaining: 43m 44s
100:	learn: 0.2418991	test: 0.2523855	best: 0.2498695 (2)	total: 6m 7s	remaining: 36m 17s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.249869464
bestIteration = 2

Shrink model to first 3 iterations.
2023: n=245,525, Brier=0.24986946, Skill=0.000522, Score=52.21, actual=0.49996, pred=0.50363
0:	learn: 0.2494284	test: 0.2498849	best: 0.2498849 (0)	total: 5.26s	remaining: 1h 1m 15s
100:	learn: 0.2429348	test: 0.2499276	best: 0.2494739 (64)	total: 7m 23s	rem

In [15]:
import numpy as np
import pandas as pd

rf = np.load("0826/results/01_rf_oof.npz")
cb = np.load("0826/results/04_features_catboost_oof.npz")

assert np.array_equal(rf["y"], cb["y"])
assert np.array_equal(rf["year"], cb["year"])

y = rf["y"].astype(float)
year = rf["year"]
rf_pred = rf["prediction"].astype(float)
cb_pred = cb["prediction"].astype(float)

def metrics(y_true, prediction):
    prediction = np.clip(prediction, 0, 1)
    rate = y_true.mean()
    brier = np.mean((prediction - y_true) ** 2)
    skill = 1 - brier / (rate * (1 - rate))
    return brier, skill, max(0, 100000 * skill)

rows = []

# rf_weight=1이면 RandomForest 단독
# rf_weight=0이면 CatBoost 단독
for rf_weight in np.arange(0, 1.01, 0.05):
    pred = rf_weight * rf_pred + (1 - rf_weight) * cb_pred

    row = {"rf_weight": round(float(rf_weight), 2)}

    for valid_year in [2022, 2023, 2024]:
        mask = year == valid_year
        _, _, score = metrics(y[mask], pred[mask])
        row[f"score_{valid_year}"] = score

    _, _, overall_score = metrics(y, pred)
    row["overall_score"] = overall_score
    row["recent_mean"] = (
        row["score_2023"] + row["score_2024"]
    ) / 2

    rows.append(row)

blend_result = pd.DataFrame(rows)
blend_result.sort_values(
    ["score_2024", "recent_mean"],
    ascending=False,
).head(10)

,rf_weight,score_2022,score_2023,score_2024,overall_score,recent_mean
19,0.95,2081.904621,0.0,416.510389,764.674185,208.255195
18,0.90,2084.414135,0.0,415.862763,788.842678,207.931382
20,1.00,2077.489630,0.0,415.573797,737.945868,207.786898
17,0.85,2085.018173,0.0,413.630918,810.451348,206.815459
16,0.80,2083.716733,0.0,409.814854,829.500196,204.907427
15,0.75,2080.509817,0.0,404.414570,845.989220,202.207285
14,0.70,2075.397425,0.0,397.430068,859.918420,198.715034
13,0.65,2068.379556,0.0,388.861348,871.287798,194.430674
12,0.60,2059.456209,0.0,378.708408,880.097352,189.354204
11,0.55,2048.627387,0.0,366.971249,886.347084,183.485624


In [16]:
rows = []

for rf_weight in np.arange(0.8, 1.001, 0.05):
    raw_pred = (
        rf_weight * rf_pred
        + (1 - rf_weight) * cb_pred
    )

    for strength in np.arange(0, 1.51, 0.1):
        row = {
            "rf_weight": round(float(rf_weight), 2),
            "strength": round(float(strength), 2),
        }

        scores = []

        for valid_year in [2023, 2024]:
            previous_mask = year == valid_year - 1
            valid_mask = year == valid_year

            # 이전 연도의 실제 평균 오차만 사용
            previous_bias = np.mean(
                y[previous_mask] - raw_pred[previous_mask]
            )

            calibrated = np.clip(
                raw_pred[valid_mask] + strength * previous_bias,
                0,
                1,
            )

            _, _, score = metrics(
                y[valid_mask],
                calibrated,
            )

            row[f"score_{valid_year}"] = score
            row[f"bias_used_{valid_year}"] = previous_bias
            scores.append(score)

        row["recent_mean"] = np.mean(scores)
        rows.append(row)

calibration_result = pd.DataFrame(rows)

calibration_result.sort_values(
    ["score_2024", "recent_mean"],
    ascending=False,
).head(20)

,rf_weight,strength,score_2023,bias_used_2023,score_2024,bias_used_2024,recent_mean
71,1.00,0.7,0,-0.001395,506.470885,-0.020304,253.235442
72,1.00,0.8,0,-0.001395,506.253535,-0.020304,253.126767
56,0.95,0.8,0,-0.001076,503.894897,-0.019473,251.947448
55,0.95,0.7,0,-0.001076,503.597065,-0.019473,251.798532
70,1.00,0.6,0,-0.001395,503.387572,-0.020304,251.693786
73,1.00,0.9,0,-0.001395,502.735523,-0.020304,251.367762
57,0.95,0.9,0,-0.001076,501.156948,-0.019473,250.578474
54,0.95,0.6,0,-0.001076,500.263453,-0.019473,250.131726
40,0.90,0.8,0,-0.000757,499.916209,-0.018641,249.958104
39,0.90,0.7,0,-0.000757,499.146446,-0.018641,249.573223
